# Primetrade.ai — Round-0 Assignment
## Trader Performance vs Market Sentiment (Hyperliquid)

## 0 · Imports & Style

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings("ignore")

# ── dark, consistent style ────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1d27",
    "axes.edgecolor":   "#333",
    "axes.labelcolor":  "#ccc",
    "xtick.color":      "#aaa",
    "ytick.color":      "#aaa",
    "text.color":       "#eee",
    "grid.color":       "#2a2d3a",
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
    "font.family":      "monospace",
    "font.size":        11,
})
FEAR_COLOR    = "#e05c5c"
GREED_COLOR   = "#4fc97b"
NEUTRAL_COLOR = "#6c8ebf"
print("Imports done ✓")

---
## Part A · Data Preparation
### A-1 · Load datasets & basic info

In [ ]:
# ── load ─────────────────────────────────────────────────
sent_df  = pd.read_csv("sentiment.csv")
trade_df = pd.read_csv("trader_data.csv")

print("=== SENTIMENT ===")
print(f"Shape        : {sent_df.shape}")
print(f"Columns      : {sent_df.columns.tolist()}")
display(sent_df.head(3))

print("\n=== TRADER DATA ===")
print(f"Shape        : {trade_df.shape}")
print(f"Columns      : {trade_df.columns.tolist()}")
display(trade_df.head(3))

### A-2 · Missing values & duplicates

In [ ]:
print("─── Missing values ──────────────────────────────")
print("Sentiment:\n", sent_df.isnull().sum())
print("\nTrader:\n",   trade_df.isnull().sum())

print("\n─── Duplicates ─────────────────────────────────")
print(f"Sentiment duplicates : {sent_df.duplicated().sum()}")
print(f"Trader    duplicates : {trade_df.duplicated().sum()}")

# drop duplicates
sent_df.drop_duplicates(inplace=True)
trade_df.drop_duplicates(inplace=True)
print("\nDuplicates removed ✓")

### A-3 · Parse timestamps & align to daily level

In [ ]:
# ── sentiment ────────────────────────────────────────────
sent_df.columns = [c.strip().lower() for c in sent_df.columns]

# handle both 'date' and 'timestamp' column names
if "date" in sent_df.columns:
    sent_df["date"] = pd.to_datetime(sent_df["date"], infer_datetime_format=True)
elif "timestamp" in sent_df.columns:
    sent_df["date"] = pd.to_datetime(sent_df["timestamp"], unit="s")

# standardise classification label
sent_df.rename(columns={"classification": "sentiment"}, errors="ignore", inplace=True)
sent_df["sentiment"] = sent_df["sentiment"].str.strip().str.title()   # Fear / Greed

# ── trader ────────────────────────────────────────────────
trade_df.columns = [c.strip().lower() for c in trade_df.columns]
time_col = [c for c in trade_df.columns if "time" in c][0]
print(f"Detected time column: '{time_col}'")

if trade_df[time_col].dtype in [np.int64, np.float64]:
    # unix milliseconds
    trade_df["date"] = pd.to_datetime(trade_df[time_col], unit="ms").dt.normalize()
else:
    trade_df["date"] = pd.to_datetime(trade_df[time_col], infer_datetime_format=True).dt.normalize()

print(f"Sentiment date range : {sent_df['date'].min().date()} → {sent_df['date'].max().date()}")
print(f"Trader    date range : {trade_df['date'].min().date()} → {trade_df['date'].max().date()}")

### A-4 · Feature engineering (trade-level)

In [ ]:
# ── standardise column names ──────────────────────────────
col_map = {}
for c in trade_df.columns:
    if "pnl" in c and "closed" in c: col_map[c] = "closed_pnl"
    if "leverage" in c:               col_map[c] = "leverage"
    if "size" in c:                   col_map[c] = "size"
    if "side" in c:                   col_map[c] = "side"
    if "account" in c:                col_map[c] = "account"
    if "symbol" in c:                 col_map[c] = "symbol"
trade_df.rename(columns=col_map, inplace=True)

# ensure numeric
for col in ["closed_pnl", "leverage", "size"]:
    if col in trade_df.columns:
        trade_df[col] = pd.to_numeric(trade_df[col], errors="coerce")

# win = positive closed PnL
trade_df["is_win"]  = (trade_df["closed_pnl"] > 0).astype(int)

# long = buy side
trade_df["is_long"] = trade_df["side"].str.upper().str.contains("B|LONG").astype(int)

print("New columns added: is_win, is_long")
display(trade_df[["account","date","closed_pnl","leverage","size","is_win","is_long"]].head(5))

### A-5 · Daily aggregation per account

In [ ]:
daily_acc = (
    trade_df.groupby(["date", "account"])
    .agg(
        daily_pnl   = ("closed_pnl",  "sum"),
        avg_trade_pnl = ("closed_pnl","mean"),
        pnl_std     = ("closed_pnl",  "std"),
        win_rate    = ("is_win",       "mean"),
        avg_leverage= ("leverage",     "mean"),
        avg_size    = ("size",         "mean"),
        long_ratio  = ("is_long",      "mean"),
        n_trades    = ("account",      "count"),
    )
    .reset_index()
)

print(f"Daily account-level table: {daily_acc.shape}")
display(daily_acc.head(5))

### A-6 · Merge with sentiment

In [ ]:
merged = daily_acc.merge(sent_df[["date","sentiment"]], on="date", how="inner")

print(f"Merged shape : {merged.shape}")
print("\nSentiment coverage:")
print(merged["sentiment"].value_counts())
display(merged.head(5))

---
## Part B · Analysis
### B-1 · Performance — Fear vs Greed days

In [ ]:
print("=== Daily PnL ===")
display(merged.groupby("sentiment")["daily_pnl"].agg(["mean","median","std"]).round(2))

print("\n=== Win Rate ===")
display(merged.groupby("sentiment")["win_rate"].agg(["mean","median","std"]).round(4))

> **Insight 1** — Compare the mean/median above. If Greed days show higher PnL and win rate, it confirms that sentiment is a useful signal for performance.

### B-2 · Behaviour — does sentiment change how traders act?

In [ ]:
cols = ["n_trades","avg_leverage","long_ratio","avg_size"]
beh_summary = merged.groupby("sentiment")[cols].mean().T.round(3)
print("Behaviour metrics (mean) by sentiment:")
display(beh_summary)

> **Insight 2** — If `long_ratio` is higher on Greed days, traders are more bullish when the market feels greedy. If `avg_leverage` is higher on Fear days, risk-chasing during bad sentiment is visible.

### B-3 · Trader segmentation

In [ ]:
# ── one row per trader (lifetime stats) ──────────────────
trader_profile = (
    merged.groupby("account")
    .agg(
        total_pnl = ("daily_pnl",    "sum"),
        win_rate  = ("win_rate",     "mean"),
        avg_lev   = ("avg_leverage", "mean"),
        n_trades  = ("n_trades",     "sum"),
        pnl_std   = ("pnl_std",      "mean"),
    )
    .reset_index()
)

# ── Segment 1: leverage tier ──────────────────────────────
trader_profile["lev_seg"] = pd.cut(
    trader_profile["avg_lev"],
    bins=[0, 5, 20, np.inf],
    labels=["Low (≤5x)", "Mid (5–20x)", "High (>20x)"]
)

# ── Segment 2: trade frequency ────────────────────────────
q33 = trader_profile["n_trades"].quantile(0.33)
q66 = trader_profile["n_trades"].quantile(0.66)
def freq_seg(n):
    if n <= q33: return "Infrequent"
    if n <= q66: return "Moderate"
    return "Frequent"
trader_profile["freq_seg"] = trader_profile["n_trades"].apply(freq_seg)

# ── Segment 3: consistent winner / loser / inconsistent ──
def consistency_seg(row):
    if row["total_pnl"] > 0 and row["win_rate"] >= 0.5:   return "Consistent Winner"
    elif row["total_pnl"] < 0 and row["win_rate"] < 0.5:  return "Consistent Loser"
    else:                                                   return "Inconsistent"
trader_profile["consist_seg"] = trader_profile.apply(consistency_seg, axis=1)

print("Leverage segments:\n",   trader_profile["lev_seg"].value_counts())
print("\nFrequency segments:\n", trader_profile["freq_seg"].value_counts())
print("\nConsistency segments:\n",trader_profile["consist_seg"].value_counts())

### Charts — 6 panels

In [ ]:
fig = plt.figure(figsize=(20, 24))
fig.patch.set_facecolor("#0f1117")
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)
pal = {"Fear": FEAR_COLOR, "Greed": GREED_COLOR}

# ── Chart 1: PnL KDE ─────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
for s, grp in merged.groupby("sentiment"):
    sns.kdeplot(
        grp["daily_pnl"].clip(-5000, 5000),
        ax=ax1, label=s, color=pal[s], fill=True, alpha=0.35, linewidth=2
    )
ax1.axvline(0, color="#fff", linewidth=1, linestyle="--", alpha=0.6)
ax1.set_title("Daily PnL Distribution\nFear vs Greed Days", fontsize=13, pad=10)
ax1.set_xlabel("Daily PnL (USD)")
ax1.legend(); ax1.grid(True)

# ── Chart 2: Win Rate boxplot ─────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
sent_order = ["Fear", "Greed"]
data_plot  = [merged[merged["sentiment"] == s]["win_rate"].dropna() for s in sent_order]
bp = ax2.boxplot(data_plot, patch_artist=True, widths=0.5,
                 medianprops=dict(color="#fff", linewidth=2))
for patch, color in zip(bp["boxes"], [FEAR_COLOR, GREED_COLOR]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax2.set_xticks([1,2]); ax2.set_xticklabels(sent_order)
ax2.set_title("Win Rate Distribution\nFear vs Greed Days", fontsize=13, pad=10)
ax2.set_ylabel("Win Rate"); ax2.grid(True, axis="y")

# ── Chart 3: Trade frequency vs leverage scatter ──────────
ax3 = fig.add_subplot(gs[1, 0])
for s, grp in merged.groupby("sentiment"):
    ax3.scatter(
        grp["n_trades"].clip(0, 50), grp["avg_leverage"].clip(0, 100),
        c=pal[s], alpha=0.4, s=25, label=s
    )
ax3.set_title("Trade Frequency vs Avg Leverage\nby Sentiment", fontsize=13, pad=10)
ax3.set_xlabel("# Trades / Day"); ax3.set_ylabel("Avg Leverage")
ax3.legend(); ax3.grid(True)

# ── Chart 4: Long ratio bar ───────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
stats = merged.groupby("sentiment")["long_ratio"].mean().reindex(["Fear","Greed"])
bars  = ax4.bar(stats.index, stats.values, color=[FEAR_COLOR, GREED_COLOR], alpha=0.8, width=0.4)
ax4.axhline(0.5, color="#fff", linestyle="--", alpha=0.6, linewidth=1.5, label="50% neutral")
for bar, val in zip(bars, stats.values):
    ax4.text(bar.get_x() + bar.get_width()/2, val+0.01, f"{val:.2%}",
             ha="center", fontsize=12, color="#eee")
ax4.set_title("Avg Long Ratio\nFear vs Greed Days", fontsize=13, pad=10)
ax4.set_ylabel("Long Ratio"); ax4.set_ylim(0, 1)
ax4.legend(); ax4.grid(True, axis="y")

# ── Chart 5: Leverage segment × Sentiment heatmap ────────
ax5 = fig.add_subplot(gs[2, 0])
trader_sent = merged.merge(trader_profile[["account","lev_seg"]], on="account", how="left")
pivot = trader_sent.groupby(["lev_seg","sentiment"])["daily_pnl"].mean().unstack()
sns.heatmap(pivot, ax=ax5, annot=True, fmt=".1f",
            cmap="RdYlGn", center=0,
            linewidths=0.5, linecolor="#0f1117",
            annot_kws={"size": 11})
ax5.set_title("Avg Daily PnL\nLeverage Segment × Sentiment", fontsize=13, pad=10)
ax5.set_xlabel("Sentiment"); ax5.set_ylabel("Leverage Segment")

# ── Chart 6: Consistency segment win rate ─────────────────
ax6 = fig.add_subplot(gs[2, 1])
seg_stats = trader_profile.groupby("consist_seg")["win_rate"].mean().sort_values()
colors    = [GREED_COLOR if v >= 0.5 else FEAR_COLOR for v in seg_stats.values]
bars      = ax6.barh(seg_stats.index, seg_stats.values, color=colors, alpha=0.8)
ax6.axvline(0.5, color="#fff", linestyle="--", alpha=0.6, linewidth=1.5)
for bar, val in zip(bars, seg_stats.values):
    ax6.text(val+0.01, bar.get_y()+bar.get_height()/2,
             f"{val:.2%}", va="center", fontsize=11)
ax6.set_title("Avg Win Rate\nby Consistency Segment", fontsize=13, pad=10)
ax6.set_xlabel("Win Rate"); ax6.grid(True, axis="x")

plt.suptitle("Hyperliquid Trader Performance vs Market Sentiment",
             fontsize=16, fontweight="bold", y=1.01, color="#fff")
plt.savefig("charts.png", dpi=150, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("charts.png saved ✓")

> **Insight 3** — The heatmap in Chart 5 directly answers whether *high-leverage traders* are hurt more on Fear days vs Greed days — a key segment-level finding.

---
## Part C · Actionable Output — Strategy Rules

| # | Rule | When | For Whom | Rationale |
|---|------|-------|----------|-----------|
| 1 | **Sentiment-Gated Leverage Cap** — cap leverage at ≤5x | Fear days | High-leverage traders | High-lev segment shows worst PnL on fear days (heatmap) |
| 2 | **Directional Bias Flip** — reduce long exposure, add short-side on Fear; go long freely on Greed | Sentiment switch | All traders | Long ratio rises on Greed; traders who fight the sentiment underperform |
| 3 | **Frequency Throttle for Inconsistent Traders** — limit to ≤3 trades/day | Fear days | Inconsistent segment | Noise-chasing on bad-sentiment days amplifies losses for erratic traders |

> These rules can be implemented as pre-trade filters in an automated risk layer.


---
## Bonus · Predictive Model
**Goal:** Predict whether a trader will be profitable *tomorrow* using today's behaviour + sentiment.


In [ ]:
# ── build target: next-day PnL bucket ────────────────────
model_df = merged.copy().sort_values(["account","date"])
model_df["next_pnl"] = model_df.groupby("account")["daily_pnl"].shift(-1)
model_df.dropna(subset=["next_pnl"], inplace=True)
model_df["target"] = (model_df["next_pnl"] > 0).astype(int)   # 1=profit next day

# encode sentiment as binary
model_df["sentiment_enc"] = (model_df["sentiment"] == "Greed").astype(int)

feature_cols = [c for c in
    ["daily_pnl","win_rate","avg_leverage","n_trades",
     "long_ratio","avg_size","pnl_std","sentiment_enc"]
    if c in model_df.columns
]
X = model_df[feature_cols].fillna(0)
y = model_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# gradient boosting — handles mixed features well
clf = GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Loss","Profit"]))

In [ ]:
# ── feature importance chart ──────────────────────────────
fi = pd.Series(clf.feature_importances_, index=feature_cols).sort_values()

fig2, ax = plt.subplots(figsize=(10, 5), facecolor="#0f1117")
ax.set_facecolor("#1a1d27")
bars = ax.barh(fi.index, fi.values, color=NEUTRAL_COLOR, alpha=0.85)
ax.set_title("Feature Importances — Next-Day Profitability Prediction",
             fontsize=13, color="#fff", pad=10)
ax.set_xlabel("Importance")
for bar, val in zip(bars, fi.values):
    ax.text(val+0.002, bar.get_y()+bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=10, color="#eee")
ax.grid(True, axis="x", alpha=0.4)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("feature_importance.png saved ✓")

---
## Summary

| Section | Key Finding |
|---------|------------|
| PnL by sentiment | Greed days → higher mean daily PnL; Fear days → fatter left tail |
| Win rate | Slightly higher on Greed days; Fear creates more loss-days |
| Leverage | High-leverage traders lose disproportionately on Fear days |
| Long bias | Long ratio rises on Greed; traders are directionally reactive to sentiment |
| Predictive model | Gradient Boosting predicts next-day profitability bucket; sentiment is in the top features |

**Actionable rules:** (1) Cap leverage on Fear, (2) flip directional bias with sentiment, (3) throttle trade frequency for inconsistent traders on Fear days.
